# ASG-WM — Colab A100 · **Runtime 1 of 2**

data &rarr; ASG labels &rarr; capacity audit &rarr; Tier-0 &rarr; Tier-1 Ph-1&ndash;Ph-3 (F1&ge;0.70 gate).
Resumable (re-run a cell after a disconnect). `device: auto` uses the A100.

> Upload your local `code/` to Drive at `MyDrive/asgwm/code`; set runtime to **A100 GPU**.


## 0 &middot; Setup


In [ ]:
# Mount Drive, point at the code, install deps, define a robust run() helper.
from google.colab import drive; drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/asgwm'        # upload your local code/ to {ROOT}/code
import sys; sys.path.insert(0, f'{ROOT}/code')
%cd {ROOT}/code
!pip -q install -r requirements.txt
!nvidia-smi -L
import subprocess
def run(cmd):
    print('$', cmd)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit('COMMAND FAILED: ' + cmd)
R = f'--override paths.root={ROOT}'
print('ROOT =', ROOT)

## 1 &middot; Config


In [ ]:
# --- edit these ---
FIRST_RUN = True   # True = time-boxed (~fits two A100 sessions). False = full publication budgets.
N_EVENTS  = 800    # SEVIR training events. Start small; raise for the final run.
# Forbid a silent synthetic fallback: a real-SEVIR failure will RAISE, not waste the session.
REAL  = '--override data.dataset=sevir --override data.require_real=true'
T1BOX = '--override train.tier1.max_steps_per_phase=4000' if FIRST_RUN else ''
T2BOX = '--override train.tier2.max_steps=12000' if FIRST_RUN else ''
print('FIRST_RUN =', FIRST_RUN, '| N_EVENTS =', N_EVENTS)

## 2 &middot; **Pre-flight (GO / NO-GO on REAL data)**
Loads a few real SEVIR events + one transition + one render on the GPU and prints **GO** or **NO-GO**. This downloads the SEVIR catalog + a couple of HDF5 files (~minutes). **Do not proceed past a NO-GO** &mdash; fix the data/source first (that is the whole point: catch problems here, not after a 12 h run).


In [ ]:
run(f'python scripts/99_preflight.py {R} {REAL}')

## 3 &middot; Data + ASG auto-labels (cached to Drive)


In [ ]:
run(f'python scripts/00_download_data.py {R} {REAL} --override data.n_train_events={N_EVENTS}')
run(f'python scripts/01_autolabel.py {R}')

## 4 &middot; Capacity audit (go/no-go)


In [ ]:
from asgwm.utils.config import load_config
from asgwm.eval.capacity import capacity_audit
print(capacity_audit(load_config('configs/default.yaml', [f'paths.root={ROOT}'])))

## 5 &middot; Tier-0 (transition + deterministic renderer + gate)


In [ ]:
run(f'python scripts/10_train_tier0.py {R}')

## 6 &middot; Tier-1 Ph-1 &rarr; Ph-3 (**hard gate F1 &ge; 0.70**)
If the gate fails the cell raises &mdash; debug the projector/data before Ph-4 (don't force it for a real run).


In [ ]:
run(f"python scripts/20_train_tier1_curriculum.py {R} {T1BOX} --override train.tier1.phases='[\"ph1_vqa\",\"ph2_desc\",\"ph3_asg\"]'")

## &check; End of Runtime 1
Checkpoints are on Drive (`MyDrive/asgwm/ckpt`). Open **Runtime 2** next. Disconnected? Re-run the affected cell &mdash; it resumes.
